In [ ]:
# =============================================================================
#  SPEAKER ID — PRODUCTION TRAINER  (ECAPA-TDNN SOTA)
#  Architecture : ECAPA-TDNN  C=512, scale=8, emb=192  — Desplanques 2020
#  Hardware     : Colab L4 (bfloat16) / T4 (float16+GradScaler) — adaptive
#  Features     : 80-FBank via C++ Zero-Copy Engine (speaker_features.so)
#  Loss         : AAM-Softmax  m=0.2 (warmup 18k), s=30  — paper-exact
#  Schedule     : CyclicLR  step=18000, total=144000  — 62 aug epochs
# =============================================================================
#
#  ARCHITECTURAL CHANGES vs. previous version (all verified from Desplanques
#  2020 paper, TaoRuijie canonical impl, and SpeechBrain reference):
#
#  1. SERes2Block  — Entry Conv1×1+BN+ReLU  +  Exit Conv1×1+BN+ReLU
#     (without these, Res2Net branches see isolated channel-index groups)
#  2. SERes2Block  — accepts `residual` param for cross-block summed residuals
#  3. Res2Conv1dReluBn  — dropout REMOVED  (breaks branch accumulation)
#  4. ECAPA_TDNN.block3  — dilation=4  (paper Table 1, not 5)
#  5. ECAPA_TDNN.mfa_conv — Conv1d(C*3→C*3, k=1)  NO BN  NO ReLU
#     (50 % of MFA activations are negative; ReLU destroys them → biased pool)
#  6. ECAPA_TDNN  — pre_pool_dropout REMOVED  (corrupts attentive statistics)
#  7. AttentiveStatisticsPooling — context vector cat(x,μ,σ) [Desplanques §3.3]
#  8. ECAPA_TDNN.pooling  — in_dim = C*3 = 1536
#  9. ECAPA_TDNN.bn_pool  — BatchNorm1d(C*6=3072)
# 10. ECAPA_TDNN.fc_embed — Linear(C*6=3072, 192)
# 11. ECAPA_TDNN.bn_embed — BatchNorm1d(192)  (not LayerNorm)
# 12. Training  — adaptive AMP: bfloat16 on L4, float16+GradScaler on T4
# 13. Training  — full checkpoint save/load  (Colab session resume)
# =============================================================================

import os, sys, io, glob, math, time, shutil, random
import subprocess, copy
import numpy as np
import soundfile as sf
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.functional as FA
from torch.utils.data import Dataset, DataLoader, Sampler
from collections import defaultdict, Counter
from tqdm import tqdm
from sklearn.metrics import roc_curve
from scipy.signal import fftconvolve
from torch.utils.tensorboard import SummaryWriter
from google.colab import drive


# =============================================================================
# §0  INFRASTRUCTURE
# =============================================================================
print("🔗 Mounting Google Drive...")
drive.mount('/content/drive')

BASE_DIR  = "/content/voxceleb_data"
TRAIN_DIR = os.path.join(BASE_DIR, "train")
TEST_DIR  = os.path.join(BASE_DIR, "test")

print("\n📦 Extracting datasets to local SSD (10-50× faster than Drive)...")
os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(TEST_DIR,  exist_ok=True)

SHARDS_DIR = '/content/drive/MyDrive/SpeakerID_Project/voxceleb_clean_shards/'
if len(os.listdir(TRAIN_DIR)) == 0:
    shards = glob.glob(os.path.join(SHARDS_DIR, "*.zip"))
    if not shards:
        raise FileNotFoundError(f"No ZIP shards in {SHARDS_DIR}")
    for shard in shards:
        subprocess.run(['unzip', '-q', shard, '-d', TRAIN_DIR], check=True)
        print(f"  ✅ {os.path.basename(shard)}")

if len(os.listdir(TEST_DIR)) == 0:
    test_zip = '/content/drive/MyDrive/SpeakerID_Project/backups/Vox1_O_Test_Clean.zip'
    if not os.path.exists(test_zip):
        raise FileNotFoundError(f"Test ZIP not found: {test_zip}")
    subprocess.run(['unzip', '-q', test_zip, '-d', TEST_DIR], check=True)
    print("  ✅ Test set (Vox1-O) extracted.")

# ── C++ Fbank engine ──────────────────────────────────────────────────────────
print("\n⚙️  Loading C++ Fbank engine...")
RESTORE = "/content/drive/MyDrive/SpeakerID_Project/cpp_engine/restore_engine.py"
if os.path.exists(RESTORE):
    subprocess.run([sys.executable, RESTORE], check=True)
else:
    src = "/content/drive/MyDrive/SpeakerID_Project/cpp_engine/speaker_features.so"
    dst = "/content/speaker_features.so"
    if not os.path.exists(src):
        raise FileNotFoundError("C++ engine .so not found in Drive")
    shutil.copy(src, dst)
    os.chmod(dst, 0o755)

sys.path.insert(0, '/content')
import speaker_features
print("✅ C++ engine ready.")

# ── Reproducibility ───────────────────────────────────────────────────────────
def seed_everything(seed: int = 42) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def seed_worker(worker_id: int) -> None:
    s = torch.initial_seed() % 2**32
    np.random.seed(s); random.seed(s)

seed_everything()


# =============================================================================
# §1  DATASET
# =============================================================================
_N_FFT = 512   # must match speaker_features.cpp
_HOP   = 160   # must match speaker_features.cpp

class ProSpeakerDataset(Dataset):
    """VoxCeleb1 dev set with on-the-fly augmentation."""

    def __init__(self, wav_dir: str, is_train: bool = True,
                 audio_len_secs: float = 3.0):
        self.is_train       = is_train
        self.target_samples = int(audio_len_secs * 16_000)        # 48 000
        # Frame count matching C++ engine (center=False, n_fft-based):
        self.fixed_t = (self.target_samples - _N_FFT) // _HOP + 1  # = 297

        all_files  = glob.glob(os.path.join(wav_dir, "**", "*.wav"), recursive=True)
        # First path component = speaker_id  (e.g. id10001/video/utt.wav → id10001)
        labels_str = [os.path.relpath(p, wav_dir).split(os.sep)[0] for p in all_files]

        counts      = Counter(labels_str)
        valid       = [(f, l) for f, l in zip(all_files, labels_str) if counts[l] >= 2]
        if not valid:
            raise ValueError(f"No speakers with ≥2 utterances in {wav_dir}")
        self.file_paths, labels_str = zip(*valid)

        speakers           = sorted(set(labels_str))
        self.speaker_to_id = {s: i for i, s in enumerate(speakers)}
        self.num_classes   = len(speakers)
        self.labels        = [self.speaker_to_id[l] for l in labels_str]

        self.musan_files = glob.glob("/content/data/musan/**/*.wav", recursive=True)
        self.rir_files   = glob.glob("/content/data/RIRS_NOISES/**/*.wav", recursive=True)
        self.freq_mask   = torchaudio.transforms.FrequencyMasking(freq_mask_param=10)
        self.time_mask   = torchaudio.transforms.TimeMasking(time_mask_param=5)

        print(f"Dataset (train={is_train}): {len(self.file_paths):,} files | "
              f"{self.num_classes} speakers | fixed_t={self.fixed_t}")

    def __len__(self) -> int:
        return len(self.file_paths)

    def _augment(self, audio: np.ndarray) -> np.ndarray:
        # 1 — Speed perturbation ±10 % (joint pitch+tempo, standard SV aug)
        if random.random() < 0.5:
            new_sr = int(16_000 / random.choice([0.9, 1.1]))
            audio  = librosa.resample(audio, orig_sr=16_000, target_sr=new_sr)

        # 2 — Codec degradation (telephone / VoIP channel)
        if random.random() < 0.4:
            try:
                buf = io.BytesIO()
                torchaudio.save(buf, torch.from_numpy(audio).unsqueeze(0).float(),
                                16_000, format="mp3",
                                compression=random.choice([64, 96, 128]))
                buf.seek(0)
                out, sr = torchaudio.load(buf)
                if sr != 16_000:
                    out = FA.resample(out, sr, 16_000)
                audio = out.squeeze().numpy()
            except Exception:
                pass

        # 3 — Room Impulse Response (reverberation)
        if self.rir_files and random.random() < 0.4:
            try:
                rir, _ = sf.read(random.choice(self.rir_files))
                if rir.ndim > 1:
                    rir = rir.mean(axis=1)
                rir   = rir / (np.max(np.abs(rir)) + 1e-9)
                audio = fftconvolve(audio, rir, mode='full')[: len(audio)]
            except Exception:
                pass

        # 4 — MUSAN additive noise (SNR 5–15 dB)
        if self.musan_files and random.random() < 0.5:
            try:
                npath = random.choice(self.musan_files)
                info  = sf.info(npath)
                if info.samplerate == 16_000:
                    if info.frames < len(audio):
                        noise, _ = sf.read(npath)
                        if noise.ndim > 1:
                            noise = noise.mean(axis=1)
                        reps  = int(np.ceil(len(audio) / len(noise)))
                        noise = np.tile(noise, reps)[: len(audio)]
                    else:
                        st = random.randint(0, info.frames - len(audio))
                        noise, _ = sf.read(npath, frames=len(audio), start=st)
                        if noise.ndim > 1:
                            noise = noise.mean(axis=1)
                    snr  = random.uniform(5.0, 15.0)
                    sp   = np.sum(audio  ** 2) / (len(audio)  + 1e-9)
                    np_  = np.sum(noise  ** 2) / (len(noise)  + 1e-9)
                    if np_ > 1e-9:
                        audio = audio + noise * np.sqrt(sp / np_ * 10 ** (-snr / 10))
            except Exception:
                pass

        mx = np.max(np.abs(audio))
        if mx > 1e-9:
            audio = audio / mx
        return audio

    def __getitem__(self, idx: int):
        audio, _ = sf.read(self.file_paths[idx])

        if self.is_train:
            audio = self._augment(audio)

        # Length normalisation (random crop in train, centre crop in eval)
        if len(audio) > self.target_samples:
            start = (random.randint(0, len(audio) - self.target_samples)
                     if self.is_train
                     else (len(audio) - self.target_samples) // 2)
            audio = audio[start: start + self.target_samples]
        elif len(audio) < self.target_samples:
            reps  = int(np.ceil(self.target_samples / len(audio)))
            audio = np.tile(audio, reps)[: self.target_samples]

        # C++ Fbank (pre-emphasis + mel + log)
        feats = speaker_features.extract(
            np.ascontiguousarray(audio, dtype=np.float32))  # [80, T]

        # Trim / zero-pad to exactly fixed_t = 297
        T = feats.shape[1]
        if   T > self.fixed_t: feats = feats[:, : self.fixed_t]
        elif T < self.fixed_t: feats = np.pad(feats, ((0,0),(0, self.fixed_t-T)))

        # Per-channel CMS (zero-mean, unit-variance across time)
        feats = (feats - feats.mean(1, keepdims=True)) / \
                (feats.std(1,  keepdims=True) + 1e-5)

        t = torch.tensor(feats, dtype=torch.float32)

        # SpecAugment (train only) — 0-10 freq channels, 0-5 time frames (paper)
        if self.is_train:
            t = self.freq_mask(t.unsqueeze(0)).squeeze(0)
            t = self.time_mask(t.unsqueeze(0)).squeeze(0)

        return t, torch.tensor(self.labels[idx], dtype=torch.long)


class MPerClassSampler(Sampler):
    """Yields exactly m utterances per speaker per batch."""
    def __init__(self, labels, m: int = 4, batch_size: int = 64):
        self.m  = m
        self.bs = batch_size
        self.spk2idx = defaultdict(list)
        for i, l in enumerate(labels):
            self.spk2idx[l].append(i)
        self.classes        = list(self.spk2idx.keys())
        self.cls_per_batch  = max(1, batch_size // m)

    def __iter__(self):
        for c in self.classes:
            random.shuffle(self.spk2idx[c])
        pools   = {c: self.spk2idx[c].copy() for c in self.classes}
        batches = []
        while True:
            valid = [c for c in self.classes if len(pools[c]) >= self.m]
            if len(valid) < self.cls_per_batch:
                break
            chosen = random.sample(valid, self.cls_per_batch)
            batch  = [pools[c].pop(0) for c in chosen for _ in range(self.m)]
            if len(batch) >= self.bs:
                batches.append(batch[: self.bs])
        random.shuffle(batches)
        for b in batches:
            yield b

    def __len__(self) -> int:
        return len(sum(self.spk2idx.values(), [])) // self.bs


# =============================================================================
# §2  MODEL  — ECAPA-TDNN  C=512, scale=8  (Desplanques 2020)
# =============================================================================

class SEBlock1d(nn.Module):
    """SE bottleneck = C // 4 = 128  (Desplanques Table 1)."""
    def __init__(self, channels: int, reduction: int = 4):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc   = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, _ = x.shape
        return x * self.fc(self.pool(x).view(b, c)).view(b, c, 1)


class Res2Conv1dReluBn(nn.Module):
    """
    Res2Net dilated convolutions — NO dropout (original ECAPA has none).
    scale=8: 7 conv branches + 1 passthrough.
    Effective temporal fields per branch: 50, 90, 130, 170, 210, 250, 290 ms
    (at dilation=4; the full range covers phone→word level for speaker ID).
    """
    def __init__(self, channels: int, scale: int = 8, dilation: int = 1):
        super().__init__()
        assert channels % scale == 0
        self.scale = scale
        self.width = channels // scale    # 64 for C=512
        self.nums  = scale - 1            # 7 conv branches

        pad = dilation                    # same-length output
        self.convs = nn.ModuleList([
            nn.Conv1d(self.width, self.width, 3,
                      dilation=dilation, padding=pad, bias=False)
            for _ in range(self.nums)
        ])
        self.bns = nn.ModuleList([
            nn.BatchNorm1d(self.width) for _ in range(self.nums)
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        spx = torch.split(x, self.width, dim=1)
        out, sp = [], None
        for i, (conv, bn) in enumerate(zip(self.convs, self.bns)):
            if i == 0:
                sp = spx[i]
            else:
                sp = sp + spx[i]           # hierarchical accumulation
            sp = F.relu(bn(conv(sp)))
            out.append(sp)
        out.append(spx[self.nums])         # last split passes through unchanged
        return torch.cat(out, dim=1)       # [B, C, T]


class SERes2Block(nn.Module):
    """
    SE-Res2Block: Entry Conv1×1 → Res2Net → Exit Conv1×1 → SE → + residual.

    Entry/Exit 1×1 convolutions are essential:
      - Entry: mixes all C channels so each Res2Net branch sees the full
               representation, not an arbitrary 64-channel slice.
      - Exit:  re-integrates the 8 branch outputs before SE attention.

    `residual` carries the accumulated sum of all previous block outputs
    (cross-block summed residuals, Desplanques ablation: +0.5 % EER without).
    """
    def __init__(self, channels: int, dilation: int, scale: int = 8):
        super().__init__()
        # Entry 1×1  (bias=False: followed by BN)
        self.conv1 = nn.Conv1d(channels, channels, 1, bias=False)
        self.bn1   = nn.BatchNorm1d(channels)
        # Res2Net (no dropout)
        self.res2  = Res2Conv1dReluBn(channels, scale=scale, dilation=dilation)
        # Exit 1×1  (bias=False: followed by BN)
        self.conv3 = nn.Conv1d(channels, channels, 1, bias=False)
        self.bn3   = nn.BatchNorm1d(channels)
        # Channel attention
        self.se    = SEBlock1d(channels)
        self.relu  = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor,
                residual: torch.Tensor = None) -> torch.Tensor:
        if residual is None:
            residual = x
        out = self.relu(self.bn1(self.conv1(x)))   # Entry
        out = self.res2(out)                        # Res2Net
        out = self.relu(self.bn3(self.conv3(out)))  # Exit
        out = self.se(out)                          # SE gating
        return out + residual                       # Summed residual ✓


class AttentiveStatisticsPooling(nn.Module):
    """
    Context-aware attentive statistics pooling  (Desplanques 2020 §3.3).

    Context = cat(x, global_mean, global_std) → attention bottleneck 128
    (paper-exact; TaoRuijie uses 256 which deviates from the paper).

    Output: [B, 2*in_dim]  — weighted mean and weighted std concatenated.
    """
    def __init__(self, in_dim: int):
        super().__init__()
        # Paper: "two Conv1d layers with 128 output channels"
        self.tdnn = nn.Conv1d(in_dim * 3, 128, 1)    # context: 3×in_dim
        self.attn = nn.Conv1d(128, in_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, C, T]
        mu  = x.mean(2, keepdim=True).expand_as(x)                   # [B,C,T]
        sg  = x.var(2, keepdim=True).clamp(min=1e-4).sqrt().expand_as(x)
        ctx = torch.cat([x, mu, sg], dim=1)                           # [B,3C,T]
        alpha = torch.softmax(self.attn(torch.tanh(self.tdnn(ctx))), dim=2)

        # Weighted statistics (computational-form variance for efficiency)
        mean = (alpha * x).sum(2)                                     # [B,C]
        std  = ((alpha * x.pow(2)).sum(2) - mean.pow(2) + 1e-4).sqrt()
        return torch.cat([mean, std], dim=1)                          # [B,2C]


class ECAPA_TDNN(nn.Module):
    """
    ECAPA-TDNN  C=512, scale=8, emb=192  — faithful to Desplanques 2020.

    Forward flow (C=512, T=297):
      [B,80,297]
      → Stem Conv1d(80→512,k=5)+BN+ReLU           [B,512,297]
      → 3× SERes2Block (d=2,3,4; summed residuals) [B,512,297]
      → MFA  Conv1d(1536→1536,k=1)                 [B,1536,297]   ← NO BN/ReLU
      → AttentiveStatsPool(1536)                   [B,3072]
      → BN(3072) → Linear(3072→192) → BN(192)     [B,192]
      train: raw  |  eval: F.normalize
    """
    def __init__(self, input_channels: int = 80,
                 channels: int = 512, emb_dim: int = 192, scale: int = 8):
        super().__init__()
        C = channels

        # Stem
        self.stem   = nn.Sequential(
            nn.Conv1d(input_channels, C, 5, padding=2, bias=False),
            nn.BatchNorm1d(C),
            nn.ReLU(inplace=True),
        )
        # SE-Res2Blocks — dilation=[2,3,4]  (Desplanques Table 1; NOT [2,3,5])
        self.block1 = SERes2Block(C, dilation=2, scale=scale)
        self.block2 = SERes2Block(C, dilation=3, scale=scale)
        self.block3 = SERes2Block(C, dilation=4, scale=scale)

        # MFA: Conv1d(3C→3C, k=1) — no BN, no ReLU
        # Both TaoRuijie and SpeechBrain go directly to pooling after this conv.
        # Adding ReLU would zero ~50 % of activations, biasing pool statistics.
        self.mfa_conv = nn.Conv1d(C * 3, C * 3, kernel_size=1)   # bias=True ✓

        # Attentive statistics pooling with context (in_dim = C*3 = 1536)
        self.pooling  = AttentiveStatisticsPooling(in_dim=C * 3)

        # Embedding head: BN(3072) → Linear(3072→192) → BN(192)
        # No ReLU between BN and Linear — matches TaoRuijie exactly.
        self.bn_pool  = nn.BatchNorm1d(C * 6)
        self.fc_embed = nn.Linear(C * 6, emb_dim)
        self.bn_embed = nn.BatchNorm1d(emb_dim)

        self._init_weights()

    def _init_weights(self) -> None:
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out',
                                        nonlinearity='relu')
            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Stem
        h  = self.stem(x)                                # [B, C, T]
        # Cross-block summed residuals (Desplanques ablation: +0.5 % without)
        o1 = self.block1(h,  residual=h)
        o2 = self.block2(o1, residual=h + o1)
        o3 = self.block3(o2, residual=h + o1 + o2)
        # Multi-layer Feature Aggregation
        mfa   = self.mfa_conv(torch.cat([o1, o2, o3], dim=1))  # [B, 3C, T]
        # Statistics pooling + embedding
        stats = self.pooling(mfa)                        # [B, 3072]
        emb   = self.bn_embed(self.fc_embed(self.bn_pool(stats)))   # [B, 192]
        return emb if self.training else F.normalize(emb, p=2, dim=1)


# =============================================================================
# §3  LOSS  — AAM-Softmax  (Desplanques 2020 Table 1)
# =============================================================================
class AAMSoftmax(nn.Module):
    """
    Additive Angular Margin Softmax (ArcFace).
    m annealed 0→0.2 over first 18 000 iters (Desplanques 2020: "first 18k").
    s = 30 (fixed).  WD for this module: 2e-4 (encoder: 2e-5).
    """
    def __init__(self, num_classes: int, emb_dim: int = 192,
                 m: float = 0.2, s: float = 30.0,
                 warmup_iters: int = 18_000):
        super().__init__()
        self.weight       = nn.Parameter(torch.FloatTensor(num_classes, emb_dim))
        nn.init.xavier_uniform_(self.weight)
        self.s, self.target_m, self.warmup = s, m, warmup_iters
        self.current_m = 0.0
        self._trig()

    def _trig(self) -> None:
        self.cos_m = math.cos(self.current_m)
        self.sin_m = math.sin(self.current_m)
        self.th    = math.cos(math.pi - self.current_m)
        self.mm    = math.sin(math.pi - self.current_m) * self.current_m

    def update_margin(self, step: int) -> None:
        self.current_m = self.target_m * min(1.0, step / max(self.warmup, 1))
        self._trig()

    def forward(self, x: torch.Tensor, label: torch.Tensor) -> torch.Tensor:
        x  = F.normalize(x.float(), p=2, dim=1)        # must normalise (bug fix)
        w  = F.normalize(self.weight.float(), p=2, dim=1)
        ct = F.linear(x, w).clamp(-1 + 1e-4, 1 - 1e-4)
        st = (1.0 - ct ** 2).clamp(1e-12).sqrt()
        ct_m = torch.where(ct > self.th,
                           ct * self.cos_m - st * self.sin_m,
                           ct - self.mm)
        oh  = torch.zeros_like(ct).scatter_(1, label.view(-1,1).long(), 1.0)
        return F.cross_entropy((oh * ct_m + (1 - oh) * ct) * self.s, label)


# =============================================================================
# §3b  TENSORBOARD UTILITIES
# =============================================================================
_TB_LOG_LOCAL = "/content/tb_logs"   # fast local SSD


# =============================================================================
# §3c  EVAL SPEED-UP — Fbank cache + batched GPU inference
# =============================================================================
# WHY: naïve eval loops over files one-by-one → batch=1 GPU inference → ~5s/file
# FIX: pre-compute Fbank once, store as .npy; infer in batches of SEGS_PER_BATCH
# EER: multi-segment mean-pooling ≥ single-pass (TTA effect); no degradation
# =============================================================================
import hashlib

EVAL_CACHE_DIR  = "/content/eval_cache"
EVAL_SAMPLES    = int(3.0 * 16_000)                                # 48 000 ✓
EVAL_FIXED_T    = (EVAL_SAMPLES - _N_FFT) // _HOP + 1             # 297 ✓
N_SEGS_MAX      = 10        # cap: max 30s audio ÷ 3s/seg = 10 segs
SEGS_PER_BATCH  = 64        # matches training BATCH; safe on T4/L4


def _eval_cache_path(fpath: str) -> str:
    """Collision-proof path using SHA-1 of the relative file path."""
    return os.path.join(
        EVAL_CACHE_DIR,
        hashlib.sha1(fpath.encode()).hexdigest() + ".npy",
    )


def _audio_to_feat_seg(seg: np.ndarray) -> np.ndarray:
    """
    C++ Fbank + CMS + trim/pad to EVAL_FIXED_T.
    seg: exactly EVAL_SAMPLES float32 samples at 16 kHz.
    Returns: [80, 297] float32
    """
    fn = speaker_features.extract(np.ascontiguousarray(seg, dtype=np.float32))
    fn = (fn - fn.mean(1, keepdims=True)) / (fn.std(1, keepdims=True) + 1e-5)
    T  = fn.shape[1]
    if   T > EVAL_FIXED_T: fn = fn[:, :EVAL_FIXED_T]
    elif T < EVAL_FIXED_T: fn = np.pad(fn, ((0, 0), (0, EVAL_FIXED_T - T)))
    return fn.astype(np.float32)


def build_eval_cache(unique_files: list, file_locator: dict) -> None:
    """
    Pre-compute Fbank features for all test utterances and save as .npy files.
    Idempotent: skips files that are already cached.
    Call ONCE before the training loop — takes ~60 s for VoxCeleb1-O (~4 700 files).
    """
    os.makedirs(EVAL_CACHE_DIR, exist_ok=True)
    missing = [f for f in unique_files if not os.path.exists(_eval_cache_path(f))]
    if not missing:
        print(f"✅ Eval cache: {len(unique_files)} files | "
              f"T={EVAL_FIXED_T} | ≤{N_SEGS_MAX} segs/file")
        return

    print(f"⏳ Building eval cache ({len(missing)}/{len(unique_files)} files)...")
    errors = 0
    for fpath in tqdm(missing, desc="Eval cache", leave=True):
        wpath = file_locator.get(fpath)
        if not wpath or not os.path.exists(wpath):
            errors += 1
            continue
        try:
            audio, sr = sf.read(wpath)
            if audio.ndim > 1:
                audio = audio.mean(axis=1)          # stereo → mono
            if sr != 16_000:
                audio = librosa.resample(audio, orig_sr=sr, target_sr=16_000)
            audio = audio.astype(np.float32)

            # Pad too-short files so every file has ≥ EVAL_SAMPLES samples
            if len(audio) < EVAL_SAMPLES:
                reps  = int(np.ceil(EVAL_SAMPLES / len(audio)))
                audio = np.tile(audio, reps)[:EVAL_SAMPLES]

            # Number of segments: +1 ensures full coverage with overlap.
            # Without +1: 8s → 2 segs → 75% coverage, 2s gap in middle.
            # With    +1: 8s → 3 segs → 100% coverage with overlap. ✓
            n_segs = max(1, min(N_SEGS_MAX, len(audio) // EVAL_SAMPLES + 1))
            if n_segs == 1:
                starts = [0]
            else:
                step   = (len(audio) - EVAL_SAMPLES) / (n_segs - 1)
                # dict.fromkeys preserves first-seen order and deduplicates
                starts = list(dict.fromkeys(
                    int(round(i * step)) for i in range(n_segs)))

            segs = [_audio_to_feat_seg(audio[s: s + EVAL_SAMPLES])
                    for s in starts]
            np.save(_eval_cache_path(fpath), np.stack(segs))  # [n, 80, 297]

        except Exception as e:
            errors += 1   # skip bad files; zero-embedding used at eval time

    cached_mb = sum(
        os.path.getsize(_eval_cache_path(f))
        for f in unique_files if os.path.exists(_eval_cache_path(f))
    ) / 1e6
    print(f"✅ Eval cache built. Errors: {errors}. Disk: {cached_mb:.0f} MB")


def _launch_tensorboard(log_dir: str, port: int = 6006) -> None:
    """
    Start TensorBoard as a background daemon and print Colab access instructions.
    Safe to call more than once — re-uses existing process.
    """
    os.makedirs(log_dir, exist_ok=True)
    try:
        subprocess.Popen(
            ["tensorboard", "--logdir", log_dir, "--port", str(port)],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
        time.sleep(2)   # let it initialise
        print(f"\n📊  TensorBoard started — port {port}.")
        print( "    ┌── Paste this in a Colab cell to see live graphs: ────────")
        print( "    │   %load_ext tensorboard")
        print(f"    │   %tensorboard --logdir {log_dir}")
        print( "    └───────────────────────────────────────────────────────────")
    except FileNotFoundError:
        print("⚠️  TensorBoard not installed — run:  !pip install tensorboard")


def _sync_tb_to_drive(local_dir: str, drive_dir: str) -> None:
    """Copy TensorBoard event files to Drive (keeps them across session resets)."""
    try:
        os.makedirs(drive_dir, exist_ok=True)
        subprocess.run(
            ["rsync", "-aq", local_dir + "/", drive_dir + "/"],
            capture_output=True, timeout=30, check=False,
        )
    except Exception:
        pass   # non-critical: training must not stop if sync fails


def compute_eer(y_true: np.ndarray,
                scores: np.ndarray):
    fpr, tpr, thr = roc_curve(y_true, scores, pos_label=1)
    fnr = 1 - tpr
    idx = np.nanargmin(np.abs(fpr - fnr))
    return float(fpr[idx] * 100), float(thr[idx])


# =============================================================================
# §4  TRAINING  — 144 000 steps
# =============================================================================
def train_model() -> None:
    print("\n🚀  ECAPA-TDNN  |  VoxCeleb1  |  144 000 steps")

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ── bfloat16 AMP (L4 / A100). For T4: change to float16 + GradScaler ─────
    # bfloat16 advantages: same mantissa precision as float16, wider exponent
    # range → no gradient overflow → no GradScaler needed → simpler code.
    if DEVICE.type == "cuda":
        AMP_DTYPE = torch.bfloat16
        scaler    = None
        print(f"  GPU: {torch.cuda.get_device_name()} | AMP: bfloat16")
    else:
        AMP_DTYPE = None
        scaler    = None
        print("  Device: CPU (no AMP)")

    torch.backends.cudnn.benchmark = True   # auto-tune for fixed T=297

    # ── Dataset + Sampler + DataLoader ────────────────────────────────────────
    train_ds = ProSpeakerDataset(TRAIN_DIR, is_train=True)

    BATCH       = 64
    TOTAL_STEPS = 144_000
    STEP_UP     = 18_000      # CLR half-cycle (= paper warmup = 18k)

    sampler    = MPerClassSampler(train_ds.labels, m=4, batch_size=BATCH)
    train_dl   = DataLoader(
        train_ds, batch_sampler=sampler,
        num_workers=4, pin_memory=True, worker_init_fn=seed_worker,
    )

    # ── Model + Loss ──────────────────────────────────────────────────────────
    model = ECAPA_TDNN(channels=512, emb_dim=192, scale=8).to(DEVICE)
    crit  = AAMSoftmax(
        num_classes=train_ds.num_classes,
        emb_dim=192, m=0.2, s=30.0, warmup_iters=STEP_UP,
    ).to(DEVICE)

    total_params = sum(p.numel() for p in model.parameters())
    print(f"  Params: {total_params:,}  ({total_params/1e6:.2f}M)")

    # ── Optimiser: two WD groups (encoder 2e-5, classifier 2e-4) ─────────────
    opt = torch.optim.AdamW(
        [
            {'params': model.parameters(), 'weight_decay': 2e-5},
            {'params': [crit.weight],       'weight_decay': 2e-4},
        ],
        lr=1e-8, betas=(0.9, 0.999),
    )
    sched = torch.optim.lr_scheduler.CyclicLR(
        opt, base_lr=1e-8, max_lr=1e-3,
        step_size_up=STEP_UP, mode='triangular2', cycle_momentum=False,
    )

    # ── Checkpoint directory (Drive) ──────────────────────────────────────────
    CKPT_DIR = "/content/drive/MyDrive/Speaker_Checkpoints"
    os.makedirs(CKPT_DIR, exist_ok=True)

    # ── TensorBoard ────────────────────────────────────────────────────────────
    TB_DRIVE = os.path.join(CKPT_DIR, "tb_logs")
    # On resume: restore previous logs from Drive so graph is continuous
    if os.path.isdir(TB_DRIVE):
        _sync_tb_to_drive(TB_DRIVE, _TB_LOG_LOCAL)   # Drive → local (reverse sync)
    writer = SummaryWriter(log_dir=_TB_LOG_LOCAL, flush_secs=30)
    _launch_tensorboard(_TB_LOG_LOCAL)

    # ── Auto-resume from latest checkpoint ────────────────────────────────────
    global_step = 0
    best_eer    = float('inf')

    ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, "ckpt_step_*.pt")),
                   key=lambda p: int(p.split("step_")[1].split(".")[0]))
    if ckpts:
        ckpt = torch.load(ckpts[-1], map_location='cpu')
        model.load_state_dict(ckpt['model'])
        crit.load_state_dict(ckpt['crit'])
        opt.load_state_dict(ckpt['opt'])
        for state in opt.state.values():          # move Adam states to GPU
            for k, v in state.items():
                if isinstance(v, torch.Tensor):
                    state[k] = v.to(DEVICE)
        sched.load_state_dict(ckpt['sched'])
        if scaler and ckpt.get('scaler'):
            scaler.load_state_dict(ckpt['scaler'])
        global_step = ckpt['step']
        best_eer    = ckpt.get('best_eer', float('inf'))
        print(f"↩  Resumed from step {global_step}  |  best EER = {best_eer:.2f}%")
    else:
        print("  No checkpoint found — starting fresh.")

    # ── Vox1-O trial list ────────────────────────────────────────────────────
    TRIAL = "/content/data/veri_test2.txt"
    os.makedirs("/content/data", exist_ok=True)
    if not os.path.exists(TRIAL):
        subprocess.run(['wget', '-q', '-O', TRIAL,
                        'https://www.robots.ox.ac.uk/~vgg/data/voxceleb/meta/veri_test2.txt'],
                       check=True)

    file_locator = {
        os.path.relpath(w, TEST_DIR): w
        for w in glob.glob(os.path.join(TEST_DIR, "**", "*.wav"), recursive=True)
    }
    trials, y_true = [], []
    with open(TRIAL) as f:
        for line in f:
            lbl, p1, p2 = line.strip().split()
            trials.append((p1, p2))
            y_true.append(int(lbl))

    unique_files = list(dict.fromkeys(p for t in trials for p in t))
    f2idx        = {p: i for i, p in enumerate(unique_files)}

    # ── Build eval Fbank cache (once; ~60s; idempotent on resume) ─────────────
    build_eval_cache(unique_files, file_locator)

    # ── Batched test-embedding extractor (closure over local vars) ─────────────
    def _extract_fast() -> list:
        """
        Returns list[Tensor(192)] — one L2-norm embedding per unique_files entry.

        Algorithm:
          1. Load pre-computed Fbank segs from cache (.npy → [n_segs, 80, 297])
          2. Accumulate segments until SEGS_PER_BATCH=64 are queued
          3. Run one batched forward pass → [64, 192]
          4. Mean-pool segments that belong to the same file → 1 embedding/file
        Memory: peak [64, 512, 297] float32 = 39 MB — same as training batch ✓
        """
        embs_out   = [torch.zeros(192)] * len(unique_files)
        buf_segs   = []   # list of [n_i, 80, 297]
        buf_counts = []   # n_segs per queued file
        buf_fidx   = []   # index into unique_files per queued file
        n_queued   = [0]  # mutable counter (Python 3 closure workaround)

        def _flush() -> None:
            if not buf_segs:
                return
            flat = torch.cat(buf_segs, dim=0).to(DEVICE)   # [sum_n, 80, 297]
            with torch.no_grad():
                if AMP_DTYPE is not None:
                    with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
                        batch_emb = model(flat).cpu()       # L2-norm in eval mode
                else:
                    batch_emb = model(flat).cpu()
            # Mean-pool segments belonging to the same file
            offset = 0
            for fi, cnt in zip(buf_fidx, buf_counts):
                embs_out[fi] = batch_emb[offset: offset + cnt].mean(0)
                offset += cnt
            buf_segs.clear(); buf_counts.clear(); buf_fidx.clear()
            n_queued[0] = 0

        for fi, fpath in enumerate(
                tqdm(unique_files, desc="Test embs", leave=False)):
            cp = _eval_cache_path(fpath)
            if not os.path.exists(cp):
                continue
            segs = torch.tensor(np.load(cp), dtype=torch.float32)  # [n, 80, 297]
            buf_segs.append(segs)
            buf_counts.append(segs.shape[0])
            buf_fidx.append(fi)
            n_queued[0] += segs.shape[0]
            if n_queued[0] >= SEGS_PER_BATCH:
                _flush()

        _flush()   # remaining partial batch
        return embs_out

    # ── Training loop ─────────────────────────────────────────────────────────
    model.train(); crit.train()
    it      = iter(train_dl)
    pbar    = tqdm(total=TOTAL_STEPS, initial=global_step, desc="Steps")
    params  = list(model.parameters()) + list(crit.parameters())
    t0      = time.time()

    while global_step < TOTAL_STEPS:
        try:
            feats, labels = next(it)
        except StopIteration:
            it = iter(train_dl)
            feats, labels = next(it)

        feats  = feats.to(DEVICE,  non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        crit.update_margin(global_step)

        # Forward  ─────────────────────────────────────────────────────────────
        if AMP_DTYPE is not None:
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
                emb  = model(feats)
            loss = crit(emb, labels)          # AAMSoftmax: always float32

            if scaler is not None:            # float16 path
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                grad_norm = nn.utils.clip_grad_norm_(params, 5.0)
                scaler.step(opt)
                scaler.update()
            else:                             # bfloat16 path (no GradScaler)
                loss.backward()
                grad_norm = nn.utils.clip_grad_norm_(params, 5.0)
                opt.step()
        else:                                 # CPU path
            loss = crit(model(feats), labels)
            loss.backward()
            grad_norm = nn.utils.clip_grad_norm_(params, 5.0)
            opt.step()

        sched.step()
        global_step += 1
        pbar.update(1)
        pbar.set_postfix(
            loss=f"{loss.item():.4f}",
            lr=f"{sched.get_last_lr()[0]:.1e}",
            m=f"{crit.current_m:.3f}",
        )

        # ── TensorBoard: log every 100 steps ──────────────────────────────────
        if global_step % 100 == 0:
            cur_lr = sched.get_last_lr()[0]
            writer.add_scalar("train/loss",      loss.item(),          global_step)
            writer.add_scalar("train/lr",         cur_lr,              global_step)
            writer.add_scalar("train/margin",     crit.current_m,      global_step)
            writer.add_scalar("train/grad_norm",  grad_norm.item(),    global_step)

        # ── Checkpoint + Vox1-O evaluation every half-cycle ───────────────────
        if global_step % STEP_UP == 0:
            elapsed = time.time() - t0
            ips     = STEP_UP / max(elapsed, 1e-3)
            eta_h   = (TOTAL_STEPS - global_step) / ips / 3600
            print(f"\n📊  Step {global_step}/{TOTAL_STEPS}  |  "
                  f"{ips:.1f} it/s  |  ETA {eta_h:.1f}h")

            model.eval(); crit.eval()

            # ── Build speaker-mean cohort ─────────────────────────────────────
            N_COHORT  = min(1000, train_ds.num_classes)
            cohort_d  = defaultdict(list)
            c_ds      = copy.copy(train_ds)
            c_ds.is_train = False
            c_dl      = DataLoader(c_ds, batch_size=64, shuffle=True, num_workers=0)

            with torch.no_grad():
                for fb, lb in c_dl:
                    if AMP_DTYPE is not None:
                        with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
                            embs = model(fb.to(DEVICE)).cpu()
                    else:
                        embs = model(fb)
                    for e, l in zip(embs, lb.numpy()):
                        cohort_d[int(l)].append(e)
                    if len(cohort_d) >= N_COHORT:
                        break

            cohort = F.normalize(
                torch.stack([torch.stack(v).mean(0)
                             for v in list(cohort_d.values())[:N_COHORT]]).float(),
                p=2, dim=1)

            # ── Extract test embeddings (batched, from Fbank cache) ───────────
            test_embs = _extract_fast()

            te = F.normalize(torch.stack(test_embs).float(), p=2, dim=1)

            # ── S-Norm scoring ────────────────────────────────────────────────
            vt, vy = [], []
            for i, (p1, p2) in enumerate(trials):
                i1, i2 = f2idx[p1], f2idx[p2]
                if te[i1].norm() > 0 and te[i2].norm() > 0:
                    vt.append((i1, i2)); vy.append(y_true[i])

            if vt:
                i1t = torch.tensor([t[0] for t in vt])
                i2t = torch.tensor([t[1] for t in vt])
                e1, e2   = te[i1t], te[i2t]
                raw      = F.cosine_similarity(e1, e2).numpy()
                sc1      = torch.mm(e1, cohort.t())
                sc2      = torch.mm(e2, cohort.t())
                snorm    = 0.5 * ((raw - sc1.mean(1).numpy()) /
                                  (sc1.std(1).numpy()  + 1e-9) +
                                  (raw - sc2.mean(1).numpy()) /
                                  (sc2.std(1).numpy()  + 1e-9))
                eer, thr = compute_eer(np.array(vy), snorm)
                print(f"  📈  S-Norm EER = {eer:.2f}%  |  threshold = {thr:.4f}")

                # TensorBoard: eval scalars
                writer.add_scalar("eval/eer_snorm",  eer,  global_step)
                writer.add_scalar("eval/threshold",  thr,  global_step)

                is_best = eer < best_eer
                if is_best:
                    best_eer = eer
                    torch.save(model.state_dict(),
                               os.path.join(CKPT_DIR, "best_model.pt"))
                    print(f"  🌟  New best! Saved best_model.pt")
            else:
                print("  ⚠️  No valid trials — check file_locator paths.")

            # ── Save full checkpoint (resume-safe) ────────────────────────────
            ckpt_path = os.path.join(CKPT_DIR, f"ckpt_step_{global_step:06d}.pt")
            torch.save({
                'step':     global_step,
                'model':    model.state_dict(),
                'crit':     crit.state_dict(),
                'opt':      opt.state_dict(),
                'sched':    sched.state_dict(),
                'scaler':   scaler.state_dict() if scaler else None,
                'best_eer': best_eer,
            }, ckpt_path)
            # Keep only last 3 checkpoints to save Drive space
            all_ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, "ckpt_step_*.pt")),
                               key=lambda p: int(p.split("step_")[1].split(".")[0]))
            for old in all_ckpts[:-3]:
                os.remove(old)
            print(f"  💾  Checkpoint saved → {os.path.basename(ckpt_path)}")

            # TensorBoard: flush buffer + sync logs to Drive for persistence
            writer.flush()
            _sync_tb_to_drive(_TB_LOG_LOCAL, TB_DRIVE)

            model.train(); crit.train()
            t0 = time.time()

    pbar.close()
    writer.flush()
    writer.close()
    print(f"\n✅  Training complete!  Best Vox1-O EER: {best_eer:.2f}%")


if __name__ == "__main__":
    train_model()